In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, random, joblib, numpy as np, pandas as pd
SHARED = '/content/drive/MyDrive/CPSC481_Closet'
D = f'{SHARED}/processed'
os.makedirs(f'{SHARED}/models', exist_ok=True)

splits = {s: json.load(open(f'{D}/{s}_cleaned.json')) for s in ['train', 'valid', 'test']}
fitb = json.load(open(f'{D}/fitb_test.json'))
ids_of = lambda o: {it['item_id'] for it in o['items']}

# Remove overlap: validation outfits may not contain training items,
# and test questions may not use training or validation items
train_ids = set().union(*(ids_of(o) for o in splits['train']))
splits['valid'] = [o for o in splits['valid'] if not ids_of(o) & train_ids]
valid_ids = set().union(*(ids_of(o) for o in splits['valid']))
seen = train_ids | valid_ids
n_fitb = len(fitb)
fitb = [q for q in fitb if not (set(q['question']) | set(q['answers'])) & seen]
print(f'kept {len(fitb)} of {n_fitb} test questions')

# Outfits as lists of item_ids, the format the later steps use
outfits = {s: [{'set_id': o['set_id'], 'items': [it['item_id'] for it in o['items']]} for o in data]
           for s, data in splits.items()}

# One row per item: item_id, split, category
items = pd.DataFrame([(it['item_id'], s, it['category'])
                      for s, data in splits.items() for o in data for it in o['items']],
                     columns=['item_id', 'split', 'category']).drop_duplicates('item_id')

# Test-question items that Sam's cleaning dropped, if any, get their category from the metadata
missing = {i for q in fitb for i in q['question'] + q['answers']} - set(items.item_id)
if missing:
    meta = json.load(open(f'{SHARED}/raw/polyvore_item_metadata.json'))
    items = pd.concat([items, pd.DataFrame([(i, 'test', meta[i].get('semantic_category')) for i in missing],
                                           columns=items.columns)], ignore_index=True)

cat = dict(zip(items.item_id, items.category))     # item_id -> category
print(len(items), 'items;', {s: len(o) for s, o in outfits.items()}, 'outfits;',
      len(fitb), 'questions;', len(missing), 'test-question items added from metadata')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
kept 1995 of 2000 test questions
113014 items; {'train': 8000, 'valid': 1274, 'test': 15145} outfits; 1995 questions; 0 test-question items added from metadata


In [3]:
def load_features():
    path = f'{D}/features.npz'
    if os.path.exists(path):
        d = np.load(path)
        return d['item_ids'].astype(str), d['image'], d['text'], False
    rng = np.random.default_rng(0)
    ids = items.item_id.to_numpy()
    img = rng.normal(size=(len(ids), 512)).astype('float32')
    txt = rng.normal(size=(len(ids), 128)).astype('float32')
    img /= np.linalg.norm(img, axis=1, keepdims=True)    # length 1, like the real ones
    txt /= np.linalg.norm(txt, axis=1, keepdims=True)
    return ids, img, txt, True

ids, IMG, TXT, fake = load_features()
row = {item_id: k for k, item_id in enumerate(ids)}    # item_id -> row number
print('PLACEHOLDER features' if fake else 'REAL features', IMG.shape, TXT.shape)

PLACEHOLDER features (113014, 512) (113014, 128)


In [4]:
def make_pairs(split, seed):
    rng = random.Random(seed)
    pool = items[items.split == split].groupby('category').item_id.apply(list).to_dict()
    pos, neg = [], []
    for o in outfits[split]:
        its = o['items']
        for i in range(len(its)):
            for j in range(i + 1, len(its)):
                a, b = its[i], its[j]
                pos.append((a, b))
                fake_b = rng.choice(pool[cat[b]])          # same category as b
                while fake_b == b:
                    fake_b = rng.choice(pool[cat[b]])
                neg.append((a, fake_b))
    return pos + neg, np.array([1] * len(pos) + [0] * len(neg))

train_pairs, y_train = make_pairs('train', seed=0)
valid_pairs, y_valid = make_pairs('valid', seed=1)
print(len(train_pairs), len(valid_pairs), y_train.mean())

181824 28714 0.5


In [5]:
def pair_matrix(img_a, txt_a, img_b, txt_b):
    """One row per pair. Shared by training and by predict_compatibility()."""
    return np.hstack([img_a * img_b, np.abs(img_a - img_b),
                      txt_a * txt_b, np.abs(txt_a - txt_b)]).astype('float32')

def pair_features(pairs):
    a = np.array([row[x] for x, _ in pairs])
    b = np.array([row[y] for _, y in pairs])
    return pair_matrix(IMG[a], TXT[a], IMG[b], TXT[b])

X_train = pair_features(train_pairs)
X_valid = pair_features(valid_pairs)
print(X_train.shape, X_valid.shape)

(181824, 1280) (28714, 1280)


In [6]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

results = {}
for C in [0.01, 0.1, 1.0]:
    m = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=1000, random_state=42))
    m.fit(X_train, y_train)
    auc = roc_auc_score(y_valid, m.predict_proba(X_valid)[:, 1])
    results[C] = (auc, m)
    print(f'C={C}: validation AUC {auc:.3f}')

best_C = max(results, key=lambda c: results[c][0])
lr_model = results[best_C][1]
joblib.dump(lr_model, f'{SHARED}/models/compat_lr.joblib')
print('best C:', best_C)

C=0.01: validation AUC 0.499
C=0.1: validation AUC 0.499
C=1.0: validation AUC 0.499
best C: 1.0


In [8]:
same = np.mean([len({cat[a] for a in q['answers']}) == 1 for q in fitb])
print(f'{same:.0%} of questions have all four candidates in one category')

fitb_rng = np.random.default_rng(0)

def fitb_accuracy(score_fn):
    right = 0
    for q in fitb:
        scores = np.array([np.mean(score_fn(q['question'], cand)) for cand in q['answers']])
        tied_best = np.flatnonzero(scores == scores.max())    # all candidates sharing the top score
        right += int(fitb_rng.choice(tied_best) == q['correct'])
    return right / len(fitb)

def model_scores(model):
    return lambda question, cand: model.predict_proba(
        pair_features([(item, cand) for item in question]))[:, 1]

print('Logistic regression FITB:', fitb_accuracy(model_scores(lr_model)))

100% of questions have all four candidates in one category
Logistic regression FITB: 0.24310776942355888


In [9]:
from collections import Counter
rng = np.random.default_rng(42)
co = Counter()
for o in outfits['train']:
    cs = [cat[i] for i in o['items']]
    for x in cs:
        for y in cs:
            if x != y:
                co[(x, y)] += 1

baselines = {
    'Random': lambda q, c: [rng.random()],
    'Category co-occurrence': lambda q, c: [co[(cat[i], cat[c])] for i in q],
    'Visual similarity': lambda q, c: [IMG[row[i]] @ IMG[row[c]] for i in q],
}
for name, fn in baselines.items():
    print(f'{name}: {fitb_accuracy(fn):.3f}')

# Similarity as a pair classifier, for comparison with step 5's AUC
sim = np.array([IMG[row[a]] @ IMG[row[b]] for a, b in valid_pairs])
print('Visual similarity AUC:', round(roc_auc_score(y_valid, sim), 3))

Random: 0.256
Category co-occurrence: 0.245
Visual similarity: 0.246
Visual similarity AUC: 0.502
